# EPI Recorder v4.4.6 — CareBridge Health Demo

**A surgical-triage bot makes a high-risk call. EPI proves exactly what happened — and judges it.**

In 6 cells you will see a real governance story, not a happy path:

1. A bot **violates policy** (clears a high-risk case with no symptom capture, no clinician signoff) → the violation is **preserved as evidence**, not hidden.
2. The fixed bot runs clean → **strict verification PASSES** and the **org trust bundle** pins it to CareBridge.
3. A forged copy **fails** verification. The evidence viewer opens with real data.

*Runtime ~3 minutes. No API key needed — the clinical model is simulated. Optionally add a `GEMINI_API_KEY` Colab Secret for live assessments.*

**Click Runtime → Run All.**

In [ ]:
# @title 1. Install EPI Recorder { display-mode: "form" }
!pip install -q epi-recorder
import epi_recorder, epi_core._version as _v
print("epi-recorder", _v.get_version())
!epi version

In [ ]:
# @title 2. Org identity + policy setup { display-mode: "form" }
# CareBridge Health as an org: a root key (offline, customer-held), a daily
# sealing key for the triage bot, and a clinical policy book.
!epi keys generate --name org-root
!epi keys generate --name triage-bot
!epi policy init --profile healthcare.triage --yes
!epi policy lint

import os
from epi_core.keys import KeyManager
from epi_core.org_bundle import fingerprint_pubkey

km = KeyManager()  # EPI_HOME-aware: newcomer keys land where you look
root_pub = km._load_public_key_raw_bytes("org-root").hex()
seal_pub = km._load_public_key_raw_bytes("triage-bot").hex()
ORG_ROOT = fingerprint_pubkey(root_pub)
os.environ["EPI_ORG_ROOT"] = ORG_ROOT  # bound into every manifest sealed below

# not_before is backdated so this bundle covers cases sealed from Jan 2025 on.
# (A bundle issued "now" cannot vouch for earlier seals — try omitting it later.)
!epi org bundle issue --bundle-id carebridge-health --root-key org-root --key triage-bot={seal_pub} --not-before 2025-01-01T00:00:00Z --out org-bundle.json
print("ORG_ROOT:", ORG_ROOT)

In [ ]:
# @title 3. The violation — bot clears a high-risk case, no checks { display-mode: "form" }
%%writefile triage_bad.py
import os
from epi_recorder import record

ORG_ROOT = os.environ["EPI_ORG_ROOT"]

# Simulated clinical model: confident, fast, and wrong.
assessment = {"risk_score": 8.4, "plan": "routine discharge"}
print(f"Model risk score: {assessment['risk_score']} -> {assessment['plan']}")

with record("bad.epi", workflow_name="Surgical Triage",
               default_key_name="triage-bot", org_root=ORG_ROOT) as epi:
    # No symptom capture. No risk tool. No clinician signoff.
    # Just the decision — exactly what the policy forbids.
    epi.log_step("agent.decision", {
        "action": "triage_decision",
        "case": "CAB-1042 (chest pain, 61y)",
        "plan": assessment["plan"],
        "clinician_signoff": None,
    })
    print("Decision recorded: routine discharge, signoff=None")

!python triage_bad.py
print("\n--- strict verification (seal is intact, signer unknown) ---")
!epi verify bad.epi --policy strict
print("\n--- audit: the violation is preserved as evidence ---")
!epi audit bad.epi --format json > audit_bad.json

import json
rep = json.load(open("audit_bad.json"))
fa = rep["pipeline"]["fault_analysis"]
print("fault_detected:", fa["fault_detected"], "| verdict:", fa["verdict"])
print("compliance:", rep["compliance_score"]["percentage"], "% ->", rep["compliance_score"]["rating"])

In [ ]:
# @title 4. The compliant run — checks done, identity pinned { display-mode: "form" }
%%writefile triage_good.py
import os
from epi_recorder import record

ORG_ROOT = os.environ["EPI_ORG_ROOT"]

with record("good.epi", workflow_name="Surgical Triage",
               default_key_name="triage-bot", org_root=ORG_ROOT) as epi:
    epi.log_step("llm.request", {"model": "triage-llm", "prompt": "intake: chest pain, 61y"})
    epi.log_step("llm.response", {"assessment": "chest pain + dyspnea, risk factors noted"})
    epi.log_step("tool.call", {"tool": "collect_symptoms", "call_id": "s1"})
    epi.log_step("tool.response", {"tool": "collect_symptoms", "call_id": "s1",
                                      "symptoms": ["chest_pain", "shortness_of_breath"]})
    epi.log_step("tool.call", {"tool": "risk_score", "call_id": "s2"})
    epi.log_step("tool.response", {"tool": "risk_score", "call_id": "s2", "risk_score": 8.4})
    epi.log_step("agent.approval.request", {"action": "clinician_signoff", "reason": "risk 8.4 > 8"})
    epi.log_step("agent.approval.response", {"action": "clinician_signoff", "approved": True,
                                             "approved_by": "dr.rao@carebridge.example"})
    epi.log_step("agent.decision", {"action": "triage_decision", "case": "CAB-1043",
                                      "plan": "escalate to clinician, priority admission"})
    print("Decision recorded: escalate, signed off by dr.rao")

!python triage_good.py
print("\n--- first look: valid seal, UNKNOWN signer (anyone can mint a key) ---")
!epi verify good.epi
print("\n--- pin the signer, then strict ---")
!epi keys trust good.epi --name triage-bot
!epi verify good.epi --policy strict
print("\n--- org binding: did this come from CareBridge? ---")
!epi org bundle verify good.epi org-bundle.json
print("\n--- full audit ---")
!epi audit good.epi

In [ ]:
# @title 5. Tamper test — flip one byte, forgery must fail { display-mode: "form" }
from pathlib import Path

original = Path("good.epi")
data = bytearray(original.read_bytes())
data[len(data) // 2] ^= 0xFF  # one bit-flip deep inside the evidence
Path("FORGED.epi").write_bytes(bytes(data))
print(f"Flipped 1 byte at offset {len(data)//2} of {len(data)}")
print("\n" + "=" * 55 + "\nORIGINAL:")
!epi verify good.epi --policy strict > /dev/null && echo "original: PASS (exit 0)"
print("\n" + "=" * 55 + "\nFORGED (1 byte changed):")
!epi verify FORGED.epi; echo "forged exit code: $?"
print("=" * 55)
Path("FORGED.epi").unlink()

In [ ]:
# @title 6. Evidence viewer — open the sealed case { display-mode: "form" }
# export-html builds a FRESH standalone viewer with current client crypto
# (no pre-verified overrides — the badge verifies for real in your browser).
!epi export-html good.epi --output triage_evidence.html

from pathlib import Path
from IPython.display import display, HTML

html_text = Path("triage_evidence.html").read_text(encoding="utf-8")
display(HTML(
    f'<div style="border:2px solid #10b981;border-radius:8px;overflow:hidden;margin:10px 0">'
    f'<div style="background:#10b981;color:white;padding:10px 16px;font-weight:bold">'
    f"EPI Evidence — good.epi (sealed, org-pinned, strict PASS)</div>"
    f'<iframe srcdoc="{html_text.replace(chr(34), chr(39))}" '
    f'width="100%" height="600" style="border:none"></iframe></div>'
))

try:
    from google.colab import files
    files.download("good.epi")
    files.download("triage_evidence.html")
    files.download("org-bundle.json")
    print("Downloaded: good.epi + triage_evidence.html + org-bundle.json")
    print("Send all three to an auditor: epi org bundle verify good.epi org-bundle.json")
except Exception:
    pass